# ResNet — Paper-to-Code Mock (Colab)

**Paper:** Deep Residual Learning for Image Recognition (He et al., 2015) — https://arxiv.org/abs/1512.03385

Mock (~30 min). Fill in the `ResidualBlock` stub, run the deep plain-vs-residual demo, then the sanity checks. This shows the **optimization / degradation** benefit on a tiny deep MLP — NOT ImageNet numbers. Reference solution in the last cell.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

## 1. Implement the residual block (YOUR TASK)

`ResidualBlock` computes `y = h + F(h)` where `F = Linear -> activation -> Linear` keeping width constant. The `PlainBlock` (same params, no skip) and `DeepNet` are given so you can race them at the same depth.

In [ ]:
class ResidualBlock(nn.Module):
    """y = h + F(h), F = Linear -> act -> Linear (same width)."""
    def __init__(self, dim, act=nn.Tanh):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.act = act()
        self.fc2 = nn.Linear(dim, dim)

    def forward(self, h):
        # TODO: compute F(h) = fc2(act(fc1(h)))
        # TODO: return the IDENTITY SKIP added to it -> h + F(h)
        raise NotImplementedError("fill me in: y = h + F(h)")


class PlainBlock(nn.Module):
    """Same params as ResidualBlock but NO skip: y = F(h)."""
    def __init__(self, dim, act=nn.Tanh):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.act = act()
        self.fc2 = nn.Linear(dim, dim)

    def forward(self, h):
        return self.fc2(self.act(self.fc1(h)))


class DeepNet(nn.Module):
    """Stem -> n_blocks of (Residual|Plain) -> head. NO BatchNorm (degradation is the point)."""
    def __init__(self, in_dim, dim, n_blocks, out_dim, residual=True, act=nn.Tanh):
        super().__init__()
        self.stem = nn.Linear(in_dim, dim)
        Block = ResidualBlock if residual else PlainBlock
        self.blocks = nn.ModuleList([Block(dim, act) for _ in range(n_blocks)])
        self.head = nn.Linear(dim, out_dim)

    def forward(self, x):
        h = torch.tanh(self.stem(x))
        for b in self.blocks:
            h = b(h)
        return self.head(h)

## 2. Demonstrate the benefit (degradation toy task)
Deep (24 blocks), narrow, saturating tanh, no BatchNorm — the regime where plain nets degrade. Plain should stall with a vanishing first-layer gradient; residual should train to low loss with a healthy gradient.

In [ ]:
def make_data(seed=0, n=512, in_dim=8):
    g = torch.Generator().manual_seed(seed)
    X = torch.randn(n, in_dim, generator=g)
    w = torch.randn(in_dim, 1, generator=g)
    y = torch.sin(X @ w) + 0.1 * (X[:, :1] ** 2)   # smooth nonlinear target
    return X, y

def train(residual, depth=24, dim=16, steps=400, seed=0):
    torch.manual_seed(seed)
    X, y = make_data(seed=0)
    net = DeepNet(8, dim, depth, 1, residual=residual, act=nn.Tanh)
    opt = torch.optim.Adam(net.parameters(), lr=3e-3)
    first_grad = None
    for step in range(steps):
        loss = F.mse_loss(net(X), y)
        opt.zero_grad(); loss.backward()
        if step == 0:
            first_grad = net.stem.weight.grad.norm().item()
        opt.step()
    return F.mse_loss(net(X), y).item(), first_grad

plain_loss, plain_g = train(residual=False)
res_loss,   res_g   = train(residual=True)
print(f"PLAIN    24-deep:  final loss {plain_loss:.4f}   first-layer grad {plain_g:.2e}")
print(f"RESIDUAL 24-deep:  final loss {res_loss:.4f}   first-layer grad {res_g:.2e}")
print(f"--> residual loss ~{plain_loss / max(res_loss, 1e-9):.0f}x lower; "
      f"first-layer grad ~{res_g / max(plain_g, 1e-12):.0g}x larger")

### Optional: plot the training curves (matplotlib — Colab only)
The numeric print above is the real evidence; this is just a visual.

In [ ]:
import matplotlib.pyplot as plt

def train_curve(residual, depth=24, dim=16, steps=400, seed=0):
    torch.manual_seed(seed)
    X, y = make_data(seed=0)
    net = DeepNet(8, dim, depth, 1, residual=residual, act=nn.Tanh)
    opt = torch.optim.Adam(net.parameters(), lr=3e-3)
    losses = []
    for _ in range(steps):
        loss = F.mse_loss(net(X), y)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    return losses

plt.plot(train_curve(False), label='plain (stalls)')
plt.plot(train_curve(True),  label='residual (trains)')
plt.xlabel('step'); plt.ylabel('train MSE'); plt.yscale('log'); plt.legend(); plt.show()

## 3. Sanity checks

In [ ]:
# 1) block output shape == input shape
blk = ResidualBlock(16); h = torch.randn(4, 16)
assert blk(h).shape == h.shape

# 2) additive: zero F's last layer -> block is identity
blk = ResidualBlock(16)
with torch.no_grad():
    blk.fc2.weight.zero_(); blk.fc2.bias.zero_()
h = torch.randn(4, 16)
assert torch.allclose(blk(h), h, atol=1e-6)

# 3) gradient flows to the first layer
net = DeepNet(8, 16, 24, 1, residual=True)
F.mse_loss(net(torch.randn(8, 8)), torch.randn(8, 1)).backward()
assert net.stem.weight.grad is not None and net.stem.weight.grad.norm().item() > 0

# 4) first-layer grad larger with residual than plain (the mechanism)
torch.manual_seed(0); netp = DeepNet(8, 16, 24, 1, residual=False)
torch.manual_seed(0); netr = DeepNet(8, 16, 24, 1, residual=True)
X, y = make_data(seed=0)
F.mse_loss(netp(X), y).backward(); F.mse_loss(netr(X), y).backward()
gp, gr = netp.stem.weight.grad.norm().item(), netr.stem.weight.grad.norm().item()
assert gr > gp

# 5) plain deep final loss > residual final loss (the demonstration)
pl, _ = train(residual=False); rl, _ = train(residual=True)
assert pl > rl

# 6) deterministic & finite in eval (no stochastic layers, no NaNs)
net = DeepNet(8, 16, 24, 1, residual=True).eval(); x = torch.randn(4, 8)
with torch.no_grad():
    a, b = net(x), net(x)
assert torch.equal(a, b) and torch.isfinite(a).all()

print("All sanity checks passed.")

## 4. Reference solution (peek only after trying)

In [ ]:
class ResidualBlockSolution(nn.Module):
    def __init__(self, dim, act=nn.Tanh):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.act = act()
        self.fc2 = nn.Linear(dim, dim)

    def forward(self, h):
        return h + self.fc2(self.act(self.fc1(h)))   # identity skip + residual F(h)